In [73]:
import pandas as pd
import os
from pybliometrics.scopus import ScopusSearch
from pybliometrics.scopus.utils import config
import numpy as np
import re


In [2]:
# Provide your key obtained from Elsevier API
config['Authentication']['APIKey']=''

In [7]:
# Set a working directory
data_directory= r""
os.chdir(data_directory)

In [8]:
#Load the sample data on journals shared on github

journals_df = pd.read_csv(
    "sample_sources.csv",
    dtype={"sourceidscopus": "Int64"}
)

In [9]:
#Add a unique journal ID due to missing details
journals_df["journal_id"] = range(1, len(journals_df) + 1)

In [10]:
journals_df

,printissn,eissn,journaltitle,sourceidscopus,journal_id
0,0022-3808,1537-534X,NaN,24404,1
1,0001-8392,NaN,Administrative Science Quarterly,<NA>,2
2,NaN,NaN,Journal of Finance,<NA>,3
3,0034-6527,1467-937X,Review of Economic Studies,24202,4
4,NaN,NaN,Journal that does not exist,<NA>,5


In [12]:
# Loop through journals and years to retrieve article metadata from Elsevier
# tries: sourceidscopus -> printissn -> eissn -> journaltitle.
# Note: as illustrated by the example data, journaltitle returns false positives.
# Note2: list of journal-years not found are saved to a separate .csv file.


years=["2020"]

def file_exists(year, ID):
    filename = f"scopusarticles_{ID}_{year}.csv"
    return os.path.isfile(filename)

# collect journals/years with no results
not_found_records = []

for j in range(len(journals_df)):
    for year in years:
        row = journals_df.loc[j]
        journal_id = int(row["journal_id"])  # stable ID for filenames + skipping

        if file_exists(year, journal_id):
            print(f"File for journal_id {journal_id} and year {year} already exists. Skipping...")
            continue

        # Build ordered search candidates: sourceidscopus -> printissn -> eissn -> journaltitle
        candidates = []

        if pd.notna(row.get("sourceidscopus")):
            candidates.append(("sourceidscopus", f"SRCID({int(row['sourceidscopus'])})"))

        if pd.notna(row.get("printissn")) and str(row["printissn"]).strip():
            candidates.append(("printissn", f"ISSN({str(row['printissn']).strip()})"))

        if pd.notna(row.get("eissn")) and str(row["eissn"]).strip():
            candidates.append(("eissn", f"ISSN({str(row['eissn']).strip()})"))

        if pd.notna(row.get("journaltitle")) and str(row["journaltitle"]).strip():
            title = str(row["journaltitle"]).strip().replace('"', '\\"')
            candidates.append(("journaltitle", f'SRCTITLE("{title}")'))

        df = None
        used_field = None
        used_query = None

        # Try each candidate until results found
        for field, base_query in candidates:
            search = f"{base_query} AND PUBYEAR IS {year}"
            print(f"Trying [{field}] -> {search}")

            s = ScopusSearch(search)

            if isinstance(s.results, list) and len(s.results) > 0:
                df = pd.DataFrame(s.results)
                used_field = field
                used_query = search
                break

        if df is not None:
            filename = f"scopusarticles_{journal_id}_{year}.csv"  # matches file_exists()
            df.to_csv(filename, index=False)
            print(f"Saved {len(df)} results to {filename} (matched on {used_field})")
        else:
            print(f"No results for journal_id {journal_id} in {year} using sourceid/issn/eissn/title.")
            
            not_found_records.append({
                "journal_id": journal_id,
                "year": year,
                "sourceidscopus": row.get("sourceidscopus"),
                "printissn": row.get("printissn"),
                "eissn": row.get("eissn"),
                "journaltitle": row.get("journaltitle")                
            })

# Save journals with no results to a separate .csv file
if not_found_records:
    not_found_df = pd.DataFrame(not_found_records)
    not_found_filename = "scopusarticles_not_found.csv"
    not_found_df.to_csv(not_found_filename, index=False)
    print(f"Saved {len(not_found_df)} not-found journal records to {not_found_filename}")
else:
    print("All journals returned results; no not-found file created.")

Trying [sourceidscopus] -> SRCID(24404) AND PUBYEAR IS 2020
Saved 108 results to scopusarticles_1_2020.csv (matched on sourceidscopus)
Trying [printissn] -> ISSN(0001-8392) AND PUBYEAR IS 2020
Saved 33 results to scopusarticles_2_2020.csv (matched on printissn)
Trying [journaltitle] -> SRCTITLE("Journal of Finance") AND PUBYEAR IS 2020
Saved 320 results to scopusarticles_3_2020.csv (matched on journaltitle)
Trying [sourceidscopus] -> SRCID(24202) AND PUBYEAR IS 2020
Saved 71 results to scopusarticles_4_2020.csv (matched on sourceidscopus)
Trying [journaltitle] -> SRCTITLE("Journal that does not exist") AND PUBYEAR IS 2020
No results for journal_id 5 in 2020 using sourceid/issn/eissn/title.
Saved 1 not-found journal records to scopusarticles_not_found.csv


In [14]:
#Function for combining all the .csv files created above into a single .csv.

def combine_journal_year_files(output_file):
    unique_publication_names = set()
    combined_df = pd.DataFrame()
    i = 0
    # Loop through all files in the directory
    for filename in os.listdir("."):       
        if filename.endswith(".csv") and re.match(r"^scopusarticles_.*_\d{4}\.csv$", filename):
            i += 1
            print(i)            
            # Load the csv file into a DataFrame
            try:
                df = pd.read_csv(filename, header=0)                
                # Append the DataFrame to the combined DataFrame                
                combined_df = pd.concat([combined_df, df], ignore_index=True)

            except Exception as e:
                print(f"Error reading file {filename}: {e}")
                continue

    # Write the combined DataFrame to a CSV file 
    combined_df.to_csv(output_file, index=False)
    
    print(f"Combined data has been written to {output_file}")

# Combine journal-year csv files and write to a single csv file
combine_journal_year_files("scopusarticles_combined.csv")

1
2
3
4
Combined data has been written to scopusarticles_combined.csv


In [16]:
# Read the combined data and print freq of missing values
df = pd.read_csv("scopusarticles_combined.csv")

missing_values = df.isnull().sum()
print("Frequency table of missing values by column:")
print(missing_values)

Frequency table of missing values by column:
eid                      0
doi                      2
pii                    527
pubmed_id              532
title                    0
subtype                  0
subtypeDescription       0
creator                  7
afid                    35
affilname               35
affiliation_city        36
affiliation_country     36
author_count             7
author_names             7
author_ids               7
author_afids            35
coverDate                0
coverDisplayDate         0
publicationName          0
issn                    27
source_id                0
eIssn                   31
aggregationType          0
volume                   8
issueIdentifier         13
article_number         512
pageRange               20
description             14
authkeywords           196
citedby_count            0
openaccess               0
freetoread             302
freetoreadLabel        302
fund_acr               346
fund_no                292
fund_spons

In [21]:
# Get a frequency table of journal titles
frequency_table = df['publicationName'].value_counts().reset_index()
frequency_table.columns = ['publicationName', 'count']
print("Frequency table of the values in the 'aggregationType' column:")
print(frequency_table)

Frequency table of the values in the 'aggregationType' column:
                                   publicationName  count
0                     Journal of Political Economy    108
1                      European Journal of Finance     96
2                               Journal of Finance     77
3                       Review of Economic Studies     71
4   International Journal of Finance and Economics     37
5                 Administrative Science Quarterly     33
6     Afro Asian Journal of Finance and Accounting     32
7                        Indian Journal of Finance     31
8    Acrn Journal of Finance and Risk Perspectives     22
9                     Quarterly Journal of Finance     20
10             Journal of Finance and Data Science      5


In [23]:
# Remove journals that were found based on the journal title but are false positives. For example:
unique_df = df[["source_id", "publicationName"]].drop_duplicates()
unique_df


,source_id,publicationName
0,24404,Journal of Political Economy
108,16036,Administrative Science Quarterly
141,14971,European Journal of Finance
145,21100862481,Acrn Journal of Finance and Risk Perspectives
146,21100244003,Indian Journal of Finance
149,21100838016,Quarterly Journal of Finance
154,17500,Journal of Finance
167,21101021576,Journal of Finance and Data Science
199,17332,International Journal of Finance and Economics
407,17500154910,Afro Asian Journal of Finance and Accounting


In [25]:
#In our case when we searched for journals with "Journal of Finance" in their title it returned false positives. 
# So we only keep those records that match exactly with the journal title.

df = df[
    ~(
        df["publicationName"].str.contains("Journal of Finance", na=False)
        & (df["publicationName"] != "Journal of Finance")
    )
]

# Print remaining journal titles

unique_df = df[["source_id", "publicationName"]].drop_duplicates()
unique_df

,source_id,publicationName
0,24404,Journal of Political Economy
108,16036,Administrative Science Quarterly
154,17500,Journal of Finance
461,24202,Review of Economic Studies


In [32]:
# Get a frequency table of the values in the 'aggregationType' column
frequency_table = df['aggregationType'].value_counts().reset_index()
frequency_table.columns = ['aggregationType', 'count']
print("Frequency table of the values in the 'aggregationType' column:")
print(frequency_table)

Frequency table of the values in the 'aggregationType' column:
  aggregationType  count
0         Journal    289


In [31]:
# Get a frequency table of the values in the 'subtypeDescription' column for "Journals"
frequency_table=df[df['aggregationType']=='Journal']['subtypeDescription'].value_counts().reset_index()
frequency_table.columns = ['subtypeDescription', 'count']
print("Frequency table of the values in the 'aggregationType' column:")
print(frequency_table)

Frequency table of the values in the 'aggregationType' column:
  subtypeDescription  count
0            Article    263
1             Review     12
2            Erratum      7
3          Editorial      3
4               Note      2
5   Conference Paper      1
6          Retracted      1


In [20]:
#Check for duplicates: same eid
duplicates = df.duplicated(subset=['eid'], keep=False)
frequency_table = df[duplicates].groupby(['eid', 'title','source_id']).size().reset_index(name='count')
print("Frequency table of duplicates based on 'eid' and 'title' columns:")
print(frequency_table)

Frequency table of duplicates based on 'eid' and 'title' columns:
Empty DataFrame
Columns: [eid, title, source_id, count]
Index: []


In [40]:
#Begin arranging and cleaning the data
articles=df.copy()
articles.shape

(289, 36)

In [41]:
articles.columns

Index(['eid', 'doi', 'pii', 'pubmed_id', 'title', 'subtype',
       'subtypeDescription', 'creator', 'afid', 'affilname',
       'affiliation_city', 'affiliation_country', 'author_count',
       'author_names', 'author_ids', 'author_afids', 'coverDate',
       'coverDisplayDate', 'publicationName', 'issn', 'source_id', 'eIssn',
       'aggregationType', 'volume', 'issueIdentifier', 'article_number',
       'pageRange', 'description', 'authkeywords', 'citedby_count',
       'openaccess', 'freetoread', 'freetoreadLabel', 'fund_acr', 'fund_no',
       'fund_sponsor'],
      dtype='object')

In [42]:
articles = articles.dropna(subset=[
    "author_ids",
    "creator",
    "afid",
    "affiliation_city",
    "affiliation_country"
])

print(len(articles), "after dropping articles that have some key variables missing")

#fix affilation name strings
articles['affilname']=articles['affilname'].astype(str).str.replace('&amp;', '')
#fix journal name strings
articles['publicationName']=articles['publicationName'].astype(str).str.replace('&amp;', 'and')

# count length of lists in below variables:
articles['authorcount']=articles['author_ids'].str.count(';')
articles['afidcount']=articles['afid'].str.count(';')
articles['affilnamecount']=articles['affilname'].str.count(';')
articles['affiliation_citycount']=articles['affiliation_city'].str.count(';')
articles['affiliation_countrycount']=articles['affiliation_country'].str.count(';')

#Check if number of coauthors in the 'author_count' column matches the listed number of coauthors in the 
# in the 'author_ids' column

articles['check0']=articles['author_count']-1==articles['authorcount'] 
articles = articles[articles['check0']==1]
print(len(articles), "after verifying consistency in num coauthors")

256 after dropping articles that have some key variables missing
256 after verifying consistency in num coauthors


In [43]:
# Frequency table of # coauthors
frequency_table = articles['author_count'].value_counts()
print(frequency_table.iloc[0:13])

author_count
2.0    89
3.0    88
1.0    44
4.0    30
5.0     5
Name: count, dtype: int64


In [44]:
#Only keep articles with at most 12 coauthors

articles=articles[articles['author_count']<13]
articles.drop(columns=['authorcount'], inplace=True)

print(len(articles), "after dropping articles that have more than 12 coauthors")

# Consistency checks

articles['check1']=articles['afidcount']==articles['affilnamecount'] 
articles['check2']=articles['afidcount']==articles['affiliation_countrycount'] 
articles['check3']=articles['afidcount']==articles['affiliation_citycount'] 

print(articles['check1'].sum(), "articles afidcount===affilnamecount")
print(articles['check2'].sum(), "articles afidcount===affilcountrycount")
print(articles['check3'].sum(), "articles afidcount===affilcitycount")

articles['check']=articles['check1']*articles['check2']*articles['check3']
articles=articles[articles['check']==True]

print(len(articles), "after dropping articles with mismatch in counts")

#seperate dataframe for article-level characteristics to be merged to final data below (excludes authors and their affiliations)
articlevars=articles[['eid', 'doi', 'coverDate', 'coverDisplayDate', 'publicationName', 'issn', 'eIssn', 'title', 'citedby_count',
                      'description', 'authkeywords', 'source_id', 'creator', 'author_count', 'aggregationType', 'volume', 
                      'issueIdentifier', 'pageRange', 'openaccess', 'freetoread', 'freetoreadLabel', 'fund_acr', 'fund_no', 'fund_sponsor']].copy()


256 after dropping articles that have more than 12 coauthors
256 articles afidcount===affilnamecount
256 articles afidcount===affilcountrycount
256 articles afidcount===affilcitycount
256 after dropping articles with mismatch in counts


In [45]:
# Save article-level varibales to a seperate .csv
articlevars.to_csv('articlevars.csv', index=False)

In [46]:
#Create article-author-affiliation level data. Note: it will only include articles where the below columns are non-missing.
data = {
    'eid': [],
    'author_names': [],
    'author_ids': [],
    'author_afids': [],
}

for _, row in articles.iterrows():
    # Split the data in each column
    author_names = str(row['author_names']).split(';') if row['author_names'] is not None else ['']
    author_ids = str(row['author_ids']).split(';') if row['author_ids'] is not None else ['']
    myid = row['eid']
    author_afids = str(row['author_afids']).split(';') if row['author_afids'] is not None else ['-99']

    # Append each piece of data to the appropriate list in the 'data' dictionary
    for author_name, author_id, author_afids in zip(author_names, author_ids, author_afids):
        # Split multiple affiliations by '-' and append new entries
        for afid in author_afids.split('-'):
            data['eid'].append(myid)
            data['author_names'].append(author_name)
            data['author_ids'].append(author_id)
            data['author_afids'].append(afid)

# Create a new DataFrame from the 'data' dictionary
authors = pd.DataFrame(data)

# Convert author_afids to integer numerics. Missing values are represented with -99

authors['author_afids']=pd.to_numeric(authors['author_afids'])
authors['author_afids'] = authors['author_afids'].fillna(-99)
authors['author_afids'] = authors['author_afids'].astype(int)
display(authors.dtypes)


eid             object
author_names    object
author_ids      object
author_afids     int32
dtype: object

In [49]:
authors
authors.to_csv('article_author_affil.csv', index=False)

In [51]:
# count the number of unique affiliations for each article and find its maximum value
# Note: an author may have multiple affiliations!

articles["semicolon_count"] = articles["afid"].str.count(";")
print(articles['semicolon_count'].max())

7


In [55]:
df_affils=articles[['afid', 'affilname', 'affiliation_city', 'affiliation_country']].copy()
df_affils

,afid,affilname,affiliation_city,affiliation_country
0,60030162;60005455,Columbia University;Yale University,New York;New Haven,United States;United States
1,60033340;60029241;60020337;115293737;100328176,Congressional Budget Office;University of Cali...,"Washington, D.C.;Santa Barbara;Cambridge;Bonn;...",United States;United States;United States;Finl...
2,60016247,Brandeis University,Waltham,United States
3,60030162;60026479,Columbia University;University of Exeter,New York;Exeter,United States;United Kingdom
4,60021784;60006297,New York University;University of Pennsylvania,New York;Philadelphia,United States;United States
...,...,...,...,...
527,60109587;60016621;60014439,Banco de España;Centre for Economic Policy Res...,Madrid;London;Davis,Spain;United Kingdom;United States
528,60108957;60025070;60000239,Ecole d’Économie de Paris;École des Hautes Étu...,Paris;Paris;Lausanne,France;France;Switzerland
529,60021406;60003858;125791554;101454543,International Monetary Fund;Uppsala Universite...,"Washington, D.C.;Uppsala;;",United States;Sweden;United States;United States
530,60007363;60005455,Northwestern University;Yale University,Evanston;New Haven,United States;United States


In [67]:
# Create an affiliation-level dataset

df_affils = articles[
    ["afid", "affilname", "affiliation_city", "affiliation_country"]
].copy()

split_cols = ["afid", "affilname", "affiliation_city", "affiliation_country"]

df_affiliations = pd.concat(
    {
        col: df_affils[col].str.split(";", expand=True).stack()
        for col in split_cols
    },
    axis=1
).reset_index()

df_affiliations = df_affiliations.rename(columns={"level_0": "id", "level_1": "rank_in_list"})
df_affiliations = df_affiliations.rename(columns={
    "affilname": "afname",
    "affiliation_city": "afcity",
    "affiliation_country": "afcountry"
})

df_affiliations['afid']=pd.to_numeric(df_affiliations['afid'])

print(len(df_affiliations))

591


In [68]:
df_affiliations=df_affiliations.drop_duplicates(subset=['afid'])
df_affiliations

,id,rank_in_list,afid,afname,afcity,afcountry
0,0,0,60030162,Columbia University,New York,United States
1,0,1,60005455,Yale University,New Haven,United States
2,1,0,60033340,Congressional Budget Office,"Washington, D.C.",United States
3,1,1,60029241,"University of California, Santa Barbara",Santa Barbara,United States
4,1,2,60020337,National Bureau of Economic Research,Cambridge,United States
...,...,...,...,...,...,...
582,528,1,60025070,École des Hautes Études en Sciences Sociales,Paris,France
584,529,0,60021406,International Monetary Fund,"Washington, D.C.",United States
586,529,2,125791554,IIES and CEPR,,United States
587,529,3,101454543,CEPR and NBER,,United States


In [71]:
df_affiliations=df_affiliations[['afid', 'afname', 'afcity', 'afcountry']]
df_affiliations.to_csv('affiliations.csv', index=False)

In [72]:
#Merge all the above to an article-author-affiliation level dataset (may not be practical for large samples)

df_merged=authors.join(df_affiliations.set_index('afid'), on='author_afids').join(articlevars.set_index('eid'), on='eid')
df_merged

,eid,author_names,author_ids,author_afids,afname,afcity,afcountry,doi,coverDate,coverDisplayDate,...,aggregationType,volume,issueIdentifier,pageRange,openaccess,freetoread,freetoreadLabel,fund_acr,fund_no,fund_sponsor
0,2-s2.0-85094597546,"Halac, Marina",55255319400,60005455,Yale University,New Haven,United States,10.1086/710560,2020-12-01,December 2020,...,Journal,128.0,12,4523-4573,0,NaN,NaN,NaN,undefined,NaN
1,2-s2.0-85094597546,"Yared, Pierre",14071002400,60030162,Columbia University,New York,United States,10.1086/710560,2020-12-01,December 2020,...,Journal,128.0,12,4523-4573,0,NaN,NaN,NaN,undefined,NaN
2,2-s2.0-85094587909,"Benzarti, Youssef",57206720405,60029241,"University of California, Santa Barbara",Santa Barbara,United States,10.1086/710558,2020-12-01,December 2020,...,Journal,128.0,12,4438-4474,0,repositoryam,Green,FEE,undefined,Foundation for Economic Education
3,2-s2.0-85094587909,"Benzarti, Youssef",57206720405,60020337,National Bureau of Economic Research,Cambridge,United States,10.1086/710558,2020-12-01,December 2020,...,Journal,128.0,12,4438-4474,0,repositoryam,Green,FEE,undefined,Foundation for Economic Education
4,2-s2.0-85094587909,"Carloni, Dorian",57206720570,60033340,Congressional Budget Office,"Washington, D.C.",United States,10.1086/710558,2020-12-01,December 2020,...,Journal,128.0,12,4438-4474,0,repositoryam,Green,FEE,undefined,Foundation for Economic Education
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
676,2-s2.0-85101200807,"ÖBERG, ERIK",57220810518,60003858,Uppsala Universitet,Uppsala,Sweden,10.1093/RESTUD/RDY060,2020-01-01,2020,...,Journal,87.0,1,77-101,0,NaN,NaN,NaN,undefined,NaN
677,2-s2.0-85101192464,"Berger, David",23476538200,60007363,Northwestern University,Evanston,United States,10.1093/RESTUD/RDZ010,2020-01-01,2020,...,Journal,87.0,1,40-76,0,repositoryvor,Green,NaN,undefined,NaN
678,2-s2.0-85101192464,"Dew-Becker, Ian",12797164900,60007363,Northwestern University,Evanston,United States,10.1093/RESTUD/RDZ010,2020-01-01,2020,...,Journal,87.0,1,40-76,0,repositoryvor,Green,NaN,undefined,NaN
679,2-s2.0-85101192464,"Giglio, Stefano",55163371700,60005455,Yale University,New Haven,United States,10.1093/RESTUD/RDZ010,2020-01-01,2020,...,Journal,87.0,1,40-76,0,repositoryvor,Green,NaN,undefined,NaN
